In [0]:
# access legecy tables
customers = spark.read.table('workspace.madstage.customers')
accounts = spark.read.table('workspace.madstage.accounts')
transactions = spark.read.table('workspace.madstage.transactions')

# look up table
customer_lookup = transactions.alias('t').join(accounts.alias('a'),on='account_id',how='left').join(customers.alias('c'),on='customer_id',how='left').select("c.customer_id","c.customer_name","c.city","c.customer_segment","c.email","c.phone",'a.account_id','a.account_type','a.branch_city','a.status','t.transaction_id','t.transaction_type','t.amount','t.transaction_date','c.is_current','c.updated_customer_dt','c.created_customer_dt')

customer_lookup.createOrReplaceTempView('customer_lookup')

# Intitalize the table
# customer_lookup.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madstage.customer_lookup')

# for incremental loads
spark.sql("""
          merge into workspace.madstage.customer_lookup target
          using customer_lookup as source
          on target.transaction_id = source.transaction_id 
          when matched then
          update set
           target.customer_id = source.customer_id,
           target.customer_name = source.customer_name,
           target.city = source.city,
           target.customer_segment = source.customer_segment,
           target.email = source.email,
           target.phone = source.phone,
           target.account_id = source.account_id,
           target.account_type = source.account_type,
           target.branch_city = source.branch_city,
           target.status = source.status,
           target.transaction_type = source.transaction_type,
           target.amount = source.amount,
           target.transaction_date = source.transaction_date,
           target.is_current = source.is_current,
           target.updated_customer_dt = source.updated_customer_dt,
           target.created_customer_dt = source.created_customer_dt
          when not matched then
          insert *
          """)


print('lookup table updated')
